# Deep Dive 3 — Gen AI Governance, End to End

**Exam domain:** Gen AI Governance · **Weight:** 28%

## The problem this solves

You know each control on its own. Then a ticket arrives that reads: *"the pipeline works when Priya
runs it and returns nothing on the schedule"*, or *"this user can call `AI_CLASSIFY` but not
`AI_AGG`"*, and knowing the controls individually does not help. Governance questions are almost
always **ordering** questions in disguise — which gate failed, and why that gate rather than another.

This is a capstone. It assumes you have worked through the four notebooks in Domain 3.0 and puts them
back together as one evaluation order you can debug against.

## What you will be able to do

- Diagnose a Cortex failure by mapping the error to the gate that produced it
- Design a least-privilege Cortex posture for several teams and defend the trade-offs
- Explain why owner's rights make a scheduled job behave differently from your session
- Choose the right cost view and the right quality metric for a given question, and say what each cannot tell you

## Before you start

This deep dive assumes the four Domain 3.0 notebooks:

- [3.1 — Set Up Model Access Controls](../Domain%203.0%20-%20Gen%20AI%20Governance/3.1.ipynb)
- [3.2 — Grant and Revoke RBAC & Privileges](../Domain%203.0%20-%20Gen%20AI%20Governance/3.2.ipynb)
- [3.3 — Manage, Monitor, and Optimize Costs](../Domain%203.0%20-%20Gen%20AI%20Governance/3.3.ipynb)
- [3.4 — Use Snowflake AI Observability Tools](../Domain%203.0%20-%20Gen%20AI%20Governance/3.4.ipynb)

You also need `ACCOUNTADMIN` and the `GENAI_STUDY` database from the repo setup script.

📖 **Snowflake documentation for this notebook**
- [AISQL privileges and model access](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)
- [SNOWFLAKE database roles](https://docs.snowflake.com/en/sql-reference/snowflake-db-roles)
- [Cross-region inference](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cross-region-inference)
- [Opting out of Snowflake AI features](https://docs.snowflake.com/en/user-guide/snowflake-cortex/opting-out)
- [Cortex Search overview](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview)
- [AI Observability reference](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-observability/reference)

---
## 1. The gate stack

Every Cortex call passes through five gates, in this order. The first one that fails is the error you
see.

```
  1. ACCOUNT PRIVILEGE      USE AI FUNCTIONS ON ACCOUNT
                            (or USE AI FUNCTION <name> ON ACCOUNT — an OR relationship)
              |  fail -> insufficient privileges on the function
              v
  2. DATABASE ROLE          CORTEX_USER  or  AI_FUNCTIONS_USER
                            (feature roles: CORTEX_ANALYST_USER, CORTEX_AGENT_USER,
                             CORTEX_EMBED_USER, CORTEX_REST_API_USER, COPILOT_USER)
              |  fail -> the same error, a different cause.
              |           AI_AGG fails here under AI_FUNCTIONS_USER.
              v
  3. MODEL ACCESS           model RBAC application role  OR  CORTEX_MODELS_ALLOWLIST
              |  fail -> a model-specific access error
              v
  4. REGION ROUTING         CORTEX_ENABLED_CROSS_REGION
              |  fail -> the model is not available in an allowed region
              v
  5. DATA AND OBJECT ACCESS USAGE on database/schema, SELECT on tables and semantic views,
                            READ on stages, USAGE on services, agents and the warehouse
                 fail -> object does not exist, or not authorized
```

**The diagnostic rule.** The error text tells you the gate. A function-level privilege error is gate 1
or 2. A model-specific error is gate 3 or 4. An object-does-not-exist error is gate 5 — and in
Snowflake, "does not exist" is what insufficient privileges on an object usually looks like, which is
why gate 5 sends people hunting for typos.

→ [More on AISQL privileges and model access](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)


---
## 2. Gates 1 and 2 — privilege and role

### The two-part rule
> Every Cortex AI function call needs **`USE AI FUNCTIONS ON ACCOUNT`** *and* **one of**
> `SNOWFLAKE.CORTEX_USER` / `SNOWFLAKE.AI_FUNCTIONS_USER`.

Both defaults are permissive, which is why least-privilege work starts with revocation rather than
with granting.

| Role / privilege | Covers | `PUBLIC` by default? |
|---|---|---|
| `USE AI FUNCTIONS` (account privilege) | every AI function | **Yes** |
| `USE AI FUNCTION <name>` (per function) | one function; OR with the blanket privilege | No |
| `SNOWFLAKE.CORTEX_USER` | AI functions **and** Cortex services | **Yes** |
| `SNOWFLAKE.AI_FUNCTIONS_USER` | **scalar** functions only — no `AI_AGG`, no `AI_SUMMARIZE_AGG`, no services | No |
| `SNOWFLAKE.CORTEX_EMBED_USER` | `AI_EMBED`, `AI_MULTI_EMBED`, `EMBED_TEXT_768`, `EMBED_TEXT_1024` | No |
| `SNOWFLAKE.CORTEX_ANALYST_USER` | Cortex Analyst only | No |
| `SNOWFLAKE.CORTEX_AGENT_USER` | Cortex Agents API only | No |
| `SNOWFLAKE.CORTEX_REST_API_USER` | the Cortex REST API only | No |
| `SNOWFLAKE.COPILOT_USER` | Cortex Code in Snowsight | **Yes** |

> **The classic scenario:** *"the user can run `AI_CLASSIFY` but `AI_AGG` fails"*. They hold
> `AI_FUNCTIONS_USER`, which excludes the two aggregate functions. Grant `CORTEX_USER` — and accept
> that you have just handed them the Cortex services too, which is the trade-off that made someone
> choose `AI_FUNCTIONS_USER` in the first place.

→ [More on the SNOWFLAKE database roles](https://docs.snowflake.com/en/sql-reference/snowflake-db-roles)


In [ ]:
%%sql
-- Least privilege starts by closing the defaults
USE ROLE ACCOUNTADMIN;
REVOKE USE AI FUNCTIONS ON ACCOUNT         FROM ROLE PUBLIC;
REVOKE DATABASE ROLE SNOWFLAKE.CORTEX_USER  FROM ROLE PUBLIC;
REVOKE DATABASE ROLE SNOWFLAKE.COPILOT_USER FROM ROLE PUBLIC;

-- Then grant deliberately: a scalar-only role
CREATE ROLE IF NOT EXISTS AI_SCALAR_ROLE;
GRANT USE AI FUNCTIONS ON ACCOUNT                TO ROLE AI_SCALAR_ROLE;
GRANT DATABASE ROLE SNOWFLAKE.AI_FUNCTIONS_USER  TO ROLE AI_SCALAR_ROLE;
-- NOTE: this role CANNOT call AI_AGG or AI_SUMMARIZE_AGG.

-- A full role
CREATE ROLE IF NOT EXISTS AI_FULL_ROLE;
GRANT USE AI FUNCTIONS ON ACCOUNT          TO ROLE AI_FULL_ROLE;
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER   TO ROLE AI_FULL_ROLE;

-- Tightest of all: one function, nothing else
CREATE ROLE IF NOT EXISTS EMBED_ONLY_ROLE;
GRANT USE AI FUNCTION AI_EMBED ON ACCOUNT        TO ROLE EMBED_ONLY_ROLE;
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_EMBED_USER  TO ROLE EMBED_ONLY_ROLE;

In [ ]:
%%sql -r cortex_db_roles
-- Which Cortex database roles exist in this account?
SHOW DATABASE ROLES IN DATABASE SNOWFLAKE;

In [ ]:
%%sql -r public_posture
-- What does PUBLIC still carry? Run this before and after the revokes above.
SHOW GRANTS TO ROLE PUBLIC;

---
## 3. Gate 3 — model access, and the OR that surprises people

Two independent mechanisms, combined with **OR**:

```
access to model M is granted if
      the role holds SNOWFLAKE."CORTEX-MODEL-ROLE-<M>"   (or CORTEX-MODEL-ROLE-ALL)
   OR M matches an entry in CORTEX_MODELS_ALLOWLIST
```

| | `CORTEX_MODELS_ALLOWLIST` | Model RBAC application roles |
|---|---|---|
| Scope | account-wide, identical for everyone | per role |
| Values | `'All'` (default), `'None'`, or a comma-separated list of lowercase model names | grant or revoke per model |
| Set where | `ALTER ACCOUNT SET …` — account level only | `CALL SNOWFLAKE.MODELS.CORTEX_BASE_MODELS_REFRESH();` then `GRANT APPLICATION ROLE` |
| Status | from August 2026 the only change still permitted is setting it to `'None'`; removed later in 2026 | the mechanism to build on |

**Because the relationship is OR, the allowlist cannot take access away from a role that holds the
application role.** Setting `CORTEX_MODELS_ALLOWLIST = 'None'` does not lock anything down on its own —
it turns the allowlist path off and leaves model RBAC in sole charge. That is the intended pattern for
per-team model control, and the cost is that every new model needs a deliberate grant per team.

### The four configurations
| Allowlist | App role granted | Result |
|---|---|---|
| `'mistral-large3'` | none | mistral-large3 only, for everyone |
| `'None'` | `CORTEX-MODEL-ROLE-LLAMA3.3-70B` | that model, for that role only |
| `'mistral-large3'` | `CORTEX-MODEL-ROLE-LLAMA3.3-70B` | **both** — the OR |
| `'None'` | none | nothing |

→ [More on model access control](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)


In [ ]:
%%sql
-- Per-role model control: turn the account-wide allowlist path off
USE ROLE ACCOUNTADMIN;
ALTER ACCOUNT SET CORTEX_MODELS_ALLOWLIST = 'None';
-- From August 2026 the only change still permitted to this parameter is setting it to 'None';
-- it is removed later in 2026. Model RBAC is the mechanism to build on.


In [ ]:
%%sql -r models_refresh
-- Populate SNOWFLAKE.MODELS so the per-model application roles exist.
-- Snowflake refreshes this daily; ACCOUNTADMIN can force it.
CALL SNOWFLAKE.MODELS.CORTEX_BASE_MODELS_REFRESH();

In [ ]:
%%sql
-- Now grant models per role. This is the mechanism that actually differentiates teams.
-- Data science: the expensive models
GRANT APPLICATION ROLE SNOWFLAKE."CORTEX-MODEL-ROLE-LLAMA3.3-70B" TO ROLE AI_FULL_ROLE;

-- Embedded app: one cheap model, nothing else
GRANT APPLICATION ROLE SNOWFLAKE."CORTEX-MODEL-ROLE-LLAMA3.1-8B"  TO ROLE AI_SCALAR_ROLE;

In [ ]:
%%sql -r models_visible
-- Output is FILTERED BY THE CURRENT ROLE'S model grants — this is how you verify gate 3
SHOW CORTEX BASE MODELS IN SCHEMA SNOWFLAKE.MODELS;

In [ ]:
%%sql -r allowlist_param
-- The account-wide setting
SHOW PARAMETERS LIKE 'CORTEX_MODELS_ALLOWLIST' IN ACCOUNT;

In [ ]:
%%sql -r secondary_roles_off
-- Secondary roles hide permission problems during testing. Turn them off to test honestly.
USE SECONDARY ROLES NONE;
SELECT CURRENT_ROLE() AS testing_as, CURRENT_SECONDARY_ROLES() AS secondary;

> **Testing pitfall worth its own line.** If `ACCOUNTADMIN` is active as a *secondary* role, every
> model looks accessible and every grant looks correct. Run `USE SECONDARY ROLES NONE;` before you test
> a role's real permissions, and `USE SECONDARY ROLES ALL;` afterwards. A test that passes because of a
> secondary role is worse than no test — it produces a documented, false assurance.


---
## 4. Gate 4 — region routing

`CORTEX_ENABLED_CROSS_REGION`, set by `ACCOUNTADMIN` only, with `ALTER ACCOUNT`.

| Tier | Values |
|---|---|
| Anywhere | `ANY_REGION` |
| One cloud | `AWS_GLOBAL` · `AZURE_GLOBAL` · `GCP_GLOBAL` |
| Cloud + geography | `AWS_US` · `AWS_EU` · `AWS_APJ` · `AWS_JP` · `AWS_AU` · `AZURE_US` · `AZURE_EU` · `GCP_US` |
| Off | `DISABLED` |

Comma-separated combinations of the cloud and cloud+geography values are supported. The default is
`ANY_REGION` for new accounts in new organizations created after **9 March 2026**; other commercial
accounts inherit a same-cloud geography value such as `AWS_US` or `AZURE_EU`.

**In transit:** within one cloud provider the traffic stays on that provider's private backbone and
never touches the public internet; across providers it crosses the public internet under mutual TLS.
**No customer data is stored** at the processing region, credits are consumed in your **requesting**
region, and there are no data egress charges.

For a data-residency requirement, `AWS_EU` or `DISABLED` is the control. There is no single-region
value — cloud plus geography is as fine as it gets, and the setting is account-wide, so you cannot give
one role a different routing policy from another.

→ [More on cross-region inference](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cross-region-inference)


In [ ]:
%%sql
-- Data residency: keep inference inside AWS EU
USE ROLE ACCOUNTADMIN;
ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'AWS_EU';

In [ ]:
%%sql -r crossregion_param
-- Verify the routing policy
SHOW PARAMETERS LIKE 'CORTEX_ENABLED_CROSS_REGION' IN ACCOUNT;

---
## 5. Gate 5 — data and object access, per feature

The database role never carries the data. This is where "it works for me" becomes "it fails for them".

| Feature | Database role (one of) | Object grants it *also* needs |
|---|---|---|
| AI functions over a table | `CORTEX_USER` / `AI_FUNCTIONS_USER` | `USAGE` on database and schema, `SELECT` on the table, `USAGE` on the warehouse |
| `AI_PARSE_DOCUMENT` / `AI_EXTRACT` on files | `CORTEX_USER` | `READ` on an internal stage (`USAGE` on an external one) |
| **Cortex Search** | `CORTEX_USER` / `CORTEX_EMBED_USER` | query: `USAGE` on the service and on its database and schema. Create: `CREATE CORTEX SEARCH SERVICE` on the schema, `SELECT` on the underlying objects, `USAGE` on the refreshing warehouse, **change tracking enabled** |
| **Cortex Analyst** | `CORTEX_USER` / `CORTEX_ANALYST_USER` | `READ` (or `WRITE`) on the stage holding a YAML semantic model, `SELECT` on the tables that model names, `USAGE` on any Search service it references |
| **Semantic view** as a query target | — | `SELECT` on the **semantic view**. Base-table `SELECT` is *not* required to query it — only to create it |
| **Cortex Agents** | `CORTEX_USER` / `CORTEX_AGENT_USER` | `CREATE AGENT` on the schema to create; `USAGE` on the agent to call it; **plus privileges on every object each tool touches** |
| **SPCS service** | — | `CREATE SERVICE` on the schema, `USAGE` on the compute pool, `READ` on the stage and image repository, plus `BIND SERVICE ENDPOINT` on the account when an endpoint is public |

### Three failure modes worth memorising

**Owner's rights.** A Cortex Search service, a stored procedure and a task all run as their **owner**.
A pipeline that works interactively and produces nothing on a schedule is almost always an owner-role
grant problem — and a search service returning rows the caller could not select directly is by design,
not a bug.

**Agents inherit nothing.** `USAGE` on an agent lets you invoke it. It does not carry the semantic
view, the search service or the warehouse. An agent a user can call but whose tools they cannot reach
answers nothing.

**Agents read the user's defaults.** An agent takes its privileges from the querying user's **default
role**, not from the role active in their session, and that default role needs `USAGE` on the user's
**default warehouse**. Neither is a grant, so neither appears in `SHOW GRANTS` — which makes this the
hardest gate-5 failure to audit.

→ [More on Cortex Search privileges](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview)
→ [More on agent access requirements](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-manage)


> ### ⚠️ Common misconceptions
>
> **"The error says the object does not exist, so the object does not exist."**
> In Snowflake an object you lack privileges on and an object that is genuinely absent produce the same
> message, by design — it stops error text leaking the existence of objects you cannot see. Check the
> grant before you check the spelling.
> → [Cortex Search overview](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview)
>
> **"Two analysts with different table grants will see different rows from the search service."**
> They will see the same rows. Cortex Search services run with owner's rights, so the index is built
> and served as the owner. If audiences need different rows, they need different services — and each
> service bills serving separately, per GB per month of indexed data.
> → [Cortex Search costs](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-costs)
>
> **"If a user's active role has the grants, the agent will work."**
> An agent resolves privileges from the user's **default** role. A user who switches role in the UI and
> tests successfully in a worksheet can still get nothing from the agent, and `SHOW GRANTS` will look
> perfect the whole time.
> → [Create and manage agents](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-manage)


In [ ]:
%%sql
-- Everything an agent-driven user needs. Note how little comes from the database role.
USE ROLE ACCOUNTADMIN;
CREATE ROLE IF NOT EXISTS AGENT_CONSUMER;

GRANT USE AI FUNCTIONS ON ACCOUNT               TO ROLE AGENT_CONSUMER;
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_AGENT_USER TO ROLE AGENT_CONSUMER;

-- the agent
GRANT USAGE ON AGENT GENAI_STUDY.PUBLIC.SUPPORT_AGENT TO ROLE AGENT_CONSUMER;

-- EVERY object the agent's tools touch, granted separately
GRANT USAGE  ON DATABASE GENAI_STUDY                                   TO ROLE AGENT_CONSUMER;
GRANT USAGE  ON SCHEMA   GENAI_STUDY.PUBLIC                            TO ROLE AGENT_CONSUMER;
GRANT SELECT ON SEMANTIC VIEW GENAI_STUDY.PUBLIC.PRODUCTS_SEMANTIC     TO ROLE AGENT_CONSUMER;
GRANT USAGE  ON CORTEX SEARCH SERVICE GENAI_STUDY.PUBLIC.TICKET_SEARCH TO ROLE AGENT_CONSUMER;
GRANT USAGE  ON WAREHOUSE COMPUTE_WH                                   TO ROLE AGENT_CONSUMER;

-- An agent reads the USER's defaults, not the session's
ALTER USER DATA_ANALYST_USER SET DEFAULT_ROLE      = AGENT_CONSUMER;
ALTER USER DATA_ANALYST_USER SET DEFAULT_WAREHOUSE = COMPUTE_WH;


In [ ]:
%%sql -r agent_consumer_grants
-- Effective grants for the role — the single most useful debugging query in Domain 3
SHOW GRANTS TO ROLE AGENT_CONSUMER;

---
## 6. Data safety — redact before, guard after

Two controls, two different jobs, routinely swapped in distractors.

| Control | Acts on | Fixes |
|---|---|---|
| `AI_REDACT` | the **input** | PII reaching the model at all |
| Cortex Guard (`model_parameters => {'guardrails': TRUE}`, default `FALSE`) | the **output** | unsafe and harmful responses |
| Grounding plus a refusal instruction | the **prompt** | hallucination |
| AI Observability groundedness | after the fact | whether the grounding worked |

`AI_REDACT` takes an array of categories, not an object, and substitutes typed placeholders:

```sql
AI_REDACT( <input> [, <categories ARRAY> ] [, <return_error_details> ] [, <mode> ] )
-- 'redact' (default) -> "[NAME] called about [PHONE_NUMBER]"
-- 'detect'           -> an OBJECT with spans: [{category, start, end, text}, ...]
```

Separately, **Cortex AI Guardrails** is an account-level feature for agents, enabled through
`ALTER ACCOUNT SET AI_SETTINGS = …` with `advanced_prompt_injection`. It scans each tool's output for
indirect prompt injection, and its scans are token-metered and reported in
`SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_GUARDRAILS_USAGE_HISTORY`.

→ [More on AI_REDACT](https://docs.snowflake.com/en/sql-reference/functions/ai_redact)
→ [More on Cortex AI Guardrails](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-ai-guardrails)


In [ ]:
%%sql
-- Correct order: redact the input FIRST, then let the model see it
CREATE OR REPLACE SECURE VIEW GENAI_STUDY.PUBLIC.TICKETS_SAFE_VIEW AS
SELECT
    ticket_id, created_at, category, status, priority,
    AI_REDACT(ticket_text) AS ticket_text
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS;

GRANT SELECT ON VIEW GENAI_STUDY.PUBLIC.TICKETS_SAFE_VIEW TO ROLE AI_SCALAR_ROLE;

In [ ]:
%%sql -r pii_detect
-- Audit mode: see what PII exists without altering the text
SELECT
    ticket_id,
    AI_REDACT(ticket_text, NULL, FALSE, 'detect') AS pii_spans
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS
LIMIT 10;

In [ ]:
%%sql -r guarded_completion
-- Guard filters the OUTPUT; AI_REDACT cleans the INPUT. Both, in the right order.
-- show_details returns choices, created, model and a usage object with
-- prompt_tokens, completion_tokens and total_tokens.
SELECT AI_COMPLETE(
    model  => 'llama3.1-8b',
    prompt => 'Reply to this complaint professionally: ' || AI_REDACT(ticket_text),
    model_parameters => {'temperature': 0, 'guardrails': TRUE},
    show_details     => TRUE
) AS guarded
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS
WHERE ticket_id = 1007;


> ### 🤔 Stop and think
>
> - The five gates are all account- or role-scoped except the data grants. If you had to hand one gate
>   to a team that is not yours to run, which would you give away, and what would you need in return to
>   still be able to debug a failure?
> - Owner's rights make search services fast and simple and make row-level security someone else's
>   problem. When is duplicating a service per audience the right call, and at what index size does
>   that stop being affordable?
> - Every control here is undone by `ACCOUNTADMIN`. What does that imply about where your real
>   governance effort should go — the grants, or the monitoring that shows when someone used the role?


---
## 7. Cost governance — which view, at which grain

| Question | View | Filter / grain |
|---|---|---|
| Daily AI spend | `METERING_DAILY_HISTORY` | `SERVICE_TYPE = 'AI_SERVICES'` |
| Hourly AI spend, to locate a spike | `METERING_HISTORY` | same filter; no `CREDITS_BILLED` column here |
| Which function and which model | `CORTEX_AI_FUNCTIONS_USAGE_HISTORY` | includes `AI_PARSE_DOCUMENT`; carries `ROLE_NAMES` and `QUERY_TAG` |
| Input versus output tokens | `CORTEX_AISQL_USAGE_HISTORY` | **excludes** `AI_PARSE_DOCUMENT` |
| Analyst messages asked | `CORTEX_ANALYST_USAGE_HISTORY` | billed **per message** |
| Search: index versus traffic | `CORTEX_SEARCH_DAILY_USAGE_HISTORY` | `CONSUMPTION_TYPE` ∈ `SERVING` / `EMBED_TEXT_TOKENS` / `BATCH` |
| Search serving, hourly | `CORTEX_SEARCH_SERVING_USAGE_HISTORY` | per service |
| External app spend | `CORTEX_REST_API_USAGE_HISTORY` | carries `INFERENCE_REGION` |
| Agent spend per user or agent | `CORTEX_AGENT_USAGE_HISTORY` | `AGENT_NAME`, `USER_NAME`, `TOKEN_CREDITS` |
| Guardrail scan spend | `CORTEX_AI_GUARDRAILS_USAGE_HISTORY` | `GUARDRAILS_SIGNAL`, `TOKEN_CREDITS` |
| PTU utilisation | `CORTEX_PROVISIONED_THROUGHPUT_USAGE_HISTORY` | `PTU_COUNT`, `PTU_CREDITS` |
| Compute pools | `SNOWPARK_CONTAINER_SERVICES_HISTORY`, or `METERING_HISTORY` with `SERVICE_TYPE = 'SNOWPARK_CONTAINER_SERVICES'` | hourly credits per pool |

Across accounts, `ORGANIZATION_USAGE` carries org-wide counterparts for several of these, plus
`USAGE_IN_CURRENCY_DAILY` for the money view.

> **The control that does not work.** A resource monitor caps **warehouse** credits. Cortex serverless
> inference is metered separately, so a resource monitor will not stop a runaway AI batch — and
> `QUERY_ATTRIBUTION_HISTORY` will not attribute it either, because that view explicitly excludes costs
> for tokens processed by AI services. Control AI spend in the query — filters, truncation, model
> choice, caching — and, for agents, with orchestration budgets, resource budgets and per-user quotas.
>
> A compute pool, for its part, bills in the `IDLE`, `ACTIVE`, `STOPPING` and `RESIZING` states and
> does not bill while `STARTING` or `SUSPENDED`. An idle pool is a real line on the bill.

→ [More on METERING_DAILY_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/metering_daily_history)
→ [More on QUERY_ATTRIBUTION_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/query_attribution_history)


In [ ]:
%%sql -r gov_daily_cost
-- Daily AI spend, with the SPCS pools beside it, in one shape
SELECT
    USAGE_DATE,
    SERVICE_TYPE,
    CREDITS_USED,
    CREDITS_BILLED
FROM SNOWFLAKE.ACCOUNT_USAGE.METERING_DAILY_HISTORY
WHERE SERVICE_TYPE IN ('AI_SERVICES', 'SNOWPARK_CONTAINER_SERVICES')
  AND USAGE_DATE >= DATEADD('day', -30, CURRENT_DATE)
ORDER BY USAGE_DATE DESC, SERVICE_TYPE;


In [ ]:
%%sql -r gov_chargeback
-- Chargeback by role. Tag your pipelines and this becomes a finance-grade report.
SELECT
    r.value::VARCHAR AS role_name,
    FUNCTION_NAME,
    MODEL_NAME,
    SUM(CREDITS)     AS credits
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_FUNCTIONS_USAGE_HISTORY,
     LATERAL FLATTEN(ROLE_NAMES) r
WHERE START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
GROUP BY 1, 2, 3
ORDER BY credits DESC;

In [ ]:
%%sql
-- Object tagging for cost attribution
CREATE TAG IF NOT EXISTS GENAI_STUDY.PUBLIC.COST_CENTER
    ALLOWED_VALUES 'ai_platform', 'data_science', 'bi_analytics', 'compliance';

ALTER CORTEX SEARCH SERVICE GENAI_STUDY.PUBLIC.TICKET_SEARCH
    SET TAG GENAI_STUDY.PUBLIC.COST_CENTER = 'ai_platform';

ALTER TABLE GENAI_STUDY.PUBLIC.SUPPORT_TICKETS
    SET TAG GENAI_STUDY.PUBLIC.COST_CENTER = 'ai_platform';

In [ ]:
%%sql -r gov_tags
-- What is tagged, and with what? Object tags describe objects; they carry no credits and
-- there is no documented join key into the AI credit views, so use them for inventory and
-- policy, and use ROLE_NAMES / QUERY_TAG for AI chargeback.
SELECT TAG_NAME, TAG_VALUE, OBJECT_DATABASE, OBJECT_SCHEMA, OBJECT_NAME, DOMAIN
FROM SNOWFLAKE.ACCOUNT_USAGE.TAG_REFERENCES
WHERE TAG_NAME = 'COST_CENTER'
  AND OBJECT_DELETED IS NULL
ORDER BY OBJECT_NAME;


---
## 8. Observability — proving the controls worked

| Metric | Question | Ground truth? |
|---|---|---|
| Context Relevance | is the retrieved context relevant to the query? | no |
| Groundedness | is the response supported by the retrieved context? | no |
| Answer Relevance | does the response address the query? | no |
| **Correctness** | how aligned is the response with the ground truth? | **yes** |
| Coherence | is the response internally consistent? | no |

Cost and latency are captured per run. Everything lands in
**`SNOWFLAKE.LOCAL.AI_OBSERVABILITY_EVENTS`**; TruLens-instrumented external agents are viewed in
Snowsight under **AI & ML » Evaluations**, and Cortex Agents have their own Observability and
Evaluations tabs. Cortex Analyst requests made directly log to
`SNOWFLAKE.LOCAL.CORTEX_ANALYST_REQUESTS_RAW`.

**Reading the first two together localises the fault:**
```
context relevance LOW  + groundedness LOW  -> retrieval  (chunking, embedding model, filters)
context relevance HIGH + groundedness LOW  -> generation (hallucination: constrain the prompt)
both HIGH, users unhappy                   -> answer relevance, then correctness
```

Agent evaluations add their own system metrics — answer correctness, tool selection accuracy, tool
execution accuracy and logical consistency — because an agent can fail at planning rather than at
wording.

Privileges for observability: the `CORTEX_USER` database role, `CREATE EXTERNAL AGENT` on the schema,
`CREATE TASK` on the schema and the global `EXECUTE TASK` privilege because runs execute as tasks, and
`USE AI FUNCTIONS` on the account because the judges are themselves AI functions. Evaluation is not
free.

→ [More on the observability metrics](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-observability/reference)


In [ ]:
%%sql -r obs_events
-- Raw observability events. RECORD_TYPE is one of LOG, SPAN, SPAN_EVENT, METRIC or EVENT;
-- traces are SPAN rows and the detail sits in the attribute columns.
SELECT TIMESTAMP, RECORD_TYPE, RESOURCE_ATTRIBUTES, RECORD_ATTRIBUTES
FROM SNOWFLAKE.LOCAL.AI_OBSERVABILITY_EVENTS
WHERE TIMESTAMP >= DATEADD('day', -7, CURRENT_TIMESTAMP())
ORDER BY TIMESTAMP DESC
LIMIT 20;


---
## 9. Opting out — and the `ACCOUNTADMIN` ceiling

| Category | Mechanism |
|---|---|
| On by default | `REVOKE DATABASE ROLE SNOWFLAKE.CORTEX_USER FROM ROLE PUBLIC;` · the same for `COPILOT_USER` · `REVOKE USE AI FUNCTIONS ON ACCOUNT FROM ROLE PUBLIC;` |
| Cortex Analyst | `ALTER ACCOUNT SET ENABLE_CORTEX_ANALYST = FALSE;` (the parameter defaults to `TRUE`) |
| Embedding functions | `REVOKE DATABASE ROLE SNOWFLAKE.CORTEX_EMBED_USER FROM ROLE <r>;` |
| Fine-tuning | `REVOKE CREATE MODEL ON SCHEMA <s> FROM ROLE <r>;` |
| Provisioned Throughput | revoke the account-level `CREATE PROVISIONED THROUGHPUT` privilege |
| Models | `ALTER ACCOUNT SET CORTEX_MODELS_ALLOWLIST = 'None';` and grant no application roles |
| Regions | `ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'DISABLED';` |

> **`ACCOUNTADMIN` has complete access to every feature in the account, AI features included.**
> Revoking database roles from `PUBLIC` does not restrict it. An account parameter binds everyone
> including `ACCOUNTADMIN` — but that role can always set it back. Role revocation controls other
> people; parameters control the account; neither controls `ACCOUNTADMIN`. The real control is not
> granting the role, backed by monitoring.

→ [More on opting out](https://docs.snowflake.com/en/user-guide/snowflake-cortex/opting-out)


> ### ⚠️ Common misconceptions
>
> **"Revoking `CORTEX_USER` from `PUBLIC` is the end of the opt-out work."**
> It covers the roles that inherited it. It does not cover `ACCOUNTADMIN`, it does not cover
> `COPILOT_USER`, and it does not touch the `USE AI FUNCTIONS` account privilege, which is separately
> granted to `PUBLIC`. Stop after the first revoke and Cortex is still reachable by everyone.
> → [Opting out of Snowflake AI features](https://docs.snowflake.com/en/user-guide/snowflake-cortex/opting-out)
>
> **"Setting an account parameter is stronger than a grant, so `ENABLE_CORTEX_ANALYST = FALSE` locks it."**
> The parameter does bind `ACCOUNTADMIN`, unlike a `PUBLIC` revoke — but `ACCOUNTADMIN` can set it back
> to `TRUE` in one statement and nothing alerts you unless you built the alert. Treat it as a control
> that needs monitoring, not a lock.
> → [Cortex Analyst](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-analyst)
>
> **"A high groundedness score means the governance controls are working."**
> Groundedness says the answer follows the retrieved context. It says nothing about whether the
> retrieved context was data the user was entitled to see — that is gate 5 and owner's rights, and no
> quality metric will ever flag it.
> → [AI Observability reference](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-observability/reference)


---

## Check your understanding

Twelve questions across the whole domain, weighted toward design judgement. Answer before expanding.

**1.** A user calls `AI_CLASSIFY` successfully and `AI_AGG` fails with a privileges error. Which gate,
and what is the fix?

<details><summary>Show answer</summary>

Gate 2. They hold `SNOWFLAKE.AI_FUNCTIONS_USER`, which covers scalar AI functions and excludes `AI_AGG`
and `AI_SUMMARIZE_AGG`. Granting `SNOWFLAKE.CORTEX_USER` fixes it — and also hands them the Cortex
services, which is why someone chose the narrower role originally. Re-granting `USE AI FUNCTIONS` does
nothing: gate 1 already passed.

→ [SNOWFLAKE database roles](https://docs.snowflake.com/en/sql-reference/snowflake-db-roles)

</details>

**2.** `CORTEX_MODELS_ALLOWLIST` is `'None'` and a role can still call `llama3.3-70b`. Why?

<details><summary>Show answer</summary>

The two model-access mechanisms combine with OR. The role holds
`SNOWFLAKE."CORTEX-MODEL-ROLE-LLAMA3.3-70B"`, and an application role grants access on its own.
`'None'` turns the allowlist path off; it revokes nothing.

→ [AISQL privileges and model access](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)

</details>

**3.** Which role can set `CORTEX_ENABLED_CROSS_REGION`, and which value pins inference to AWS Europe?

<details><summary>Show answer</summary>

`ACCOUNTADMIN` only, using `ALTER ACCOUNT`. `AWS_EU` is the value. `ORGADMIN` is a distractor, and so
is any single-region value — cloud plus geography is the finest granularity the parameter offers.

→ [Cross-region inference](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cross-region-inference)

</details>

**4.** A task-driven pipeline produces nothing on its schedule and works when you run the same SQL by
hand. What do you check first, and why does it fail silently?

<details><summary>Show answer</summary>

The **task owner's** grants. Tasks run as their owner, not as whoever created or triggered them. It
fails silently because a role that cannot see rows gets zero rows, which is a valid result, not an
error. Check what the owner role holds, not what you hold.

→ [Cortex Search overview](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview)

</details>

**5.** Two analysts with different table grants query the same Cortex Search service and get identical
rows. Bug or design?

<details><summary>Show answer</summary>

Design. Search services run with owner's rights, so the index is built and served as the owner and the
caller's own row-level grants are not applied at query time. If audiences must differ, build separate
services over separately filtered sources — and pay serving credits for each index.

→ [Cortex Search overview](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview)

</details>

**6.** An agent user's `SHOW GRANTS` looks perfect and every question still fails. What two user
properties would you check?

<details><summary>Show answer</summary>

`DEFAULT_ROLE` and `DEFAULT_WAREHOUSE`. An agent resolves privileges from the querying user's default
role, and that role needs `USAGE` on the user's default warehouse. Neither is a grant, so neither
appears in `SHOW GRANTS` — which is precisely why this one survives an audit.

→ [Create and manage agents](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-manage)

</details>

**7.** A nightly `AI_COMPLETE` batch is overrunning its budget. Your colleague attaches a resource
monitor. Will it help, and what would you do instead?

<details><summary>Show answer</summary>

It will not. Resource monitors cap warehouse credits and Cortex serverless inference is metered
separately, so the monitor never trips. Cut it in the query: a smaller model, fewer rows, truncated
input, an `AI_COUNT_TOKENS` gate, and a cache column so you stop reprocessing history. Each of those
costs you something — accuracy, context or freshness — and that is the conversation to have with the
job's owner rather than a control to install quietly.

→ [METERING_DAILY_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/metering_daily_history)

</details>

**8.** You are asked to design Cortex access for three teams: data science (all models, all
functions), BI (Analyst only, one small model), and an ETL service account (embeddings only). Sketch
the design and name its ongoing cost.

<details><summary>Show answer</summary>

Revoke `CORTEX_USER`, `COPILOT_USER` and `USE AI FUNCTIONS` from `PUBLIC` first, or nothing below
matters. Then: data science gets `CORTEX_USER` + `USE AI FUNCTIONS` + `CORTEX-MODEL-ROLE-ALL`; BI gets
`CORTEX_ANALYST_USER` + `USE AI FUNCTIONS` + one model application role, plus `SELECT` on the semantic
view; the ETL account gets `CORTEX_EMBED_USER` + `USE AI FUNCTION AI_EMBED` and nothing else. Set
`CORTEX_MODELS_ALLOWLIST = 'None'` so model RBAC is the only path. The ongoing cost is that every new
model and every new capability is a deliberate grant per team, forever, and the day nobody owns that
work the design quietly stops matching reality.

→ [AISQL privileges and model access](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)

</details>

**9.** Legal asks for EU-only processing. You can set `CORTEX_ENABLED_CROSS_REGION = 'AWS_EU'` or
`'DISABLED'`. Which do you recommend, and what do you tell the teams who will be affected?

<details><summary>Show answer</summary>

`'AWS_EU'` in most cases: it satisfies the geography constraint while keeping routing available across
AWS Europe, so models absent from your own region remain reachable. `'DISABLED'` is stricter and
breaks any pipeline depending on a model that is not served locally. Either way the setting is
account-wide — you cannot exempt one team — so tell everyone which models they are about to lose
before you set it, not after.

→ [Cross-region inference](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cross-region-inference)

</details>

**10.** A chatbot's answers are "made up". Groundedness is low, context relevance is high. Where is the
fault, and what would you change first?

<details><summary>Show answer</summary>

Generation, not retrieval — the model was handed relevant context and did not use it. Constrain the
prompt with an explicit refusal instruction, keep `temperature` at 0 and add a `response_format`
schema. Rebuilding the index would spend effort on the half that already works.

→ [AI Observability reference](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-observability/reference)

</details>

**11.** Your CISO wants assurance that nobody can use Cortex without approval. Write the honest answer.

<details><summary>Show answer</summary>

You can revoke `CORTEX_USER`, `COPILOT_USER` and `USE AI FUNCTIONS` from `PUBLIC`, set
`CORTEX_MODELS_ALLOWLIST = 'None'` with no application roles granted, and pin or disable cross-region
routing — and all of that is undone by anyone holding `ACCOUNTADMIN`, who has complete access to every
AI feature regardless. The honest answer is that the control is the small number of people holding
`ACCOUNTADMIN`, plus alerting on `SERVICE_TYPE = 'AI_SERVICES'` credits and on parameter changes. Tell
them that rather than promising a lock that does not exist.

→ [Opting out of Snowflake AI features](https://docs.snowflake.com/en/user-guide/snowflake-cortex/opting-out)

</details>

**12.** Connecting across the exam: an SPCS-hosted inference service is charged to your team and you
did not deploy anything this month. Which view shows the spend, and what state is the pool most likely
in?

<details><summary>Show answer</summary>

`SNOWPARK_CONTAINER_SERVICES_HISTORY` gives hourly credits per compute pool, and `METERING_HISTORY`
with `SERVICE_TYPE = 'SNOWPARK_CONTAINER_SERVICES'` is the metering-family answer. The pool is almost
certainly `IDLE` — pools bill in `IDLE`, `ACTIVE`, `STOPPING` and `RESIZING`, and do not bill while
`STARTING` or `SUSPENDED`. Suspending it stops the charge.

→ [SNOWPARK_CONTAINER_SERVICES_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/snowpark_container_services_history)

</details>
